« model_18 — PV: NOKTA ve VEKTÖR · TINYSTORIES »

**Soru: Matematikte aritmetik vardı; dilde bağlam ve ilişki var. PV, zinciri OLDUĞU GİBİ (bütün hikâye tek toplam) bırakıldığında dili ne kadar taşıyor?**
Kullanıcı, 24 Eylül: *"matematik de çok aritmetik bir işlem var ama dil de bağlam ve ilişki matematik gibi değil. o yüzden görmek istiyorum."*

| ne | değer |
|---|---|
| veri | TinyStories V2 (GPT-4), Drive önbelleği `tam_n4000_v2` · kelime düzeyi, 4.003 token · her hikâye bir pencere, `<eos>` başta ve sonda |
| T | 512 (hikâyelerin %98,6'sı sığar) · RM 512 satır |
| PV | d 1024 · layer başına 256 vektör · aktif 8 · 4 layer · skor sözlükten: kare, S_p öğrenilen |
| koşular | `TS_PV_D1024` start randn (boy ~32), 11.268'de durduruldu: seçimi start'ların kendi boyu belirliyordu · `TS_PV_START1` start boyu 1 (`start_norm`), ~57.000'de durduruldu: hikâyenin ortası öğrenilmedi · `TS_PV_LAM09` zincir solar, `lam` 0,9: ardışık C'ler ayrışsın · `TS_PV_REL07` GÖRELİ zincir (`chain` relative), `lam` 0,7: aynı bağlam her konumda aynı nokta (eğitimsiz kNN: %13,3 → %34,8) · `TS_PV_CCACHE` REL07 + `c_cache` ve `gate`: hikâyenin kendi geçmişinden kopya (kâğıt üstü: ppl −%14, geçmiş isim %3,9 → %16), ~10.000'de durduruldu: sıcaklık 0'da döngü · `TS_PV_CCSKIP20` `cache_skip` 20: defter son 20 kelimeye bakmaz (kâğıt üstü, CCACHE t8000: döngü ölçüsü 0,28 → 0,63, ppl +%3,7), ~4.000'de durduruldu · `TS_PV_SELECT` CCACHE'in aynısı, tek fark `select` "direction": katman 1'den itibaren aktif vektörler yöne göre seçilir, taşınmış noktanın boyu seçimi ezmez (kâğıt üstü, CCACHE t10000: derin katmanlarda işin %90'ını 10 / 6 vektör yapıyor, C_m 1024 boyutun 40 yönünde; yöne bakınca aynı vektörlerle 113 yön) · ~8.500'de durduruldu: derin katmanlarda iş yapan vektör iki katına çıktı ama 256'nın 226-230'u hiç seçilmedi (top-8 kuralı) · `TS_PV_DENSE` CCACHE'in aynısı, tek fark `active` 256: nokta bütün vektörlerden geçer, 4.000 adım; seçim keskin ama bütün C'ler aynı 2-21 vektöre gidiyor (start'lar merkeze yapıştı) · `TS_PV_SELECT_LB` SELECT'in aynısı, tek fark `load_balance` 0,01: kayba "bütün C'lerde kullanım dengeli olsun" terimi, 4.000 adım: 0,4007 (SELECT'in 8.500 düzeyi), derin katmanlarda 180 / 176 / 82 vektör, son katman hâlâ 19 · `TS_PV_LB_CONTENT` SELECT_LB + `c_content`: C = [`C_order` 512 | `C_content` 512], yakın C çiftlerinin %80'inde son 4 kelime aynı (bilgi C'de yok), 4.000 adım: heldout 0,4434 / ppl 14,35, eos_ok 0,78 · `TS_PV_LB_CONTENT2` aynı ayarlar, düzeltilmiş denge (sabit sıcaklık), 20.000 adım · `TS_PV_FULL` **tepe**: bütün vektörler aktif (`active` 256, top-k yok), seçim yöne göre (`select` direction), defter `Q` ile aranır (64 ok, hepsi aktif, `C_m`'den), bütün geçmiş (`cache_topk` None); C = [`C_order` 512 | `C_content` 512] (`d_order`, `d_content`); `C_content`: kaydırmasız, `lam_w`/`beta_w` token × boyut öğrenilir (kâğıt üstü, CCACHE t10000: mükemmel arama tavanı %40,3 → %66,7; defter kapalıyken de açgözlü üretim 19/20 döngü, kök kısa bellek) |
| lr · batch · adım | 0,002 · 64 · 16.000 |
| ölçüm · kayıt | ölçüm ve ağırlık kaydı (`w`) her 500, tam yedek (`t`) her 2.000 adımda |
| ölçüt | accuracy (sonraki kelime birebir), ppl · tanı: konuma göre accuracy (`acc_0_64`, `acc_64_256`, `acc_256_512`), `eos_ok` · **METİN: modelin yazdığı hikâye (5 HİKÂYE YAZ)** |

C'nin kapasitesi (hesap, eğitim değil): d 1024'te C'ye toplanan kelimelerin 25'inde %100'ü, 50'sinde %82'si, 171'inde (medyan hikâye) %12'si geri okunabilir.

**Sıra:** `0 HAZIRLIK` → `1l KOŞU` (LB_CONTENT2, 20.000) → `3 NABIZ` · `4 EĞRİ` · `5 HİKÂYE YAZ` → bitince `6 SONUÇ`
**Çekirdek düşerse:** `0 HAZIRLIK` → `2 SÜRDÜR`

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "2 SURDUR".  Hikaye verisi ~4 GB RAM.
import os, re, sys, subprocess

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir ve SURDUR ikinci bir kopya
# baslatir (olculdu, 24 Eylul: baglanti kopunca cekirdek dustu sanildi, LB_CONTENT2 iki kez kostu).
if 'train' in sys.modules:
    _canli = [r.name for r in sys.modules['train'].RUNS.values() if r.alive]
    assert not _canli, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 3 NABIZ' % _canli

from google.colab import drive
drive.mount('/content/drive')
KOK = '/content/drive/MyDrive/model_18'
TS = '/content/drive/MyDrive/tinystories'
os.makedirs(KOK, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
DEPO = '/content/sekerai'
if os.path.isdir(DEPO):
    subprocess.run(['git', '-C', DEPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', DEPO, 'reset', '-q', '--hard', 'origin/main'],
                   check=True)
else:
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/sekerahmet/sekerai.git', DEPO],
                   check=True)
SRC = DEPO + '/deneme2/model_18'
KOD = subprocess.run(['git', '-C', DEPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip()
print('kod   ' + KOD)
assert os.path.exists(SRC + '/data_stories.py'), 'depoda data_stories YOK -- yerelde git push gerekli'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_18', 'train', 'data_stories'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_18, train
import data_stories as DS

# Kural 9: sozluk ve akis Drive onbelleginden (model_17'nin urettigi).
SOZ, (_eg, _em), (_dg, _dm) = DS.kur(TS, T=512, en=4000)      # SOZ: sozluk, <eos> dahil
IX = {a: i for i, a in enumerate(SOZ)}
N, EOS = len(SOZ), IX[DS.SON]
EG = (torch.from_numpy(_eg), torch.from_numpy(_em))
DG = (torch.from_numpy(_dg), torch.from_numpy(_dm))
del _eg, _em, _dg, _dm
OLCUT = DS.olcut(EG, DG, eos=EOS, aygit='cuda', en=2000)

ORTAK = dict(lr=0.002, batch=64, steps=16000, eval_every=500, save_every=2000,
             weights_every=500, seed=0)
# Tepe (model_18) = modelin SON YAPISI.  Kullanici, 24 Eylul: "farklı değerler kullandığımızda bu
# ayrı bir yere yazarız".  Eski kosular tepeden FARKLI her degeri burada yazar; SADE, ozellikler
# gelmeden onceki PV'dir.
SADE = dict(start_norm=None, lam=1.0, chain='absolute', c_cache=False, cache_topk=8, query=False,
            c_content=False, select='distance', active=8, load_balance=0.0)
# Kullanici, 24 Eylul: "d 1024 olsun" ve "1024 e 64 olsun".
KOSU = {'TS_PV_D1024': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512),
        # Kullanici, 24 Eylul: "Startları küçük başlat ... bunu durdur yenisi ile koş".
        'TS_PV_START1': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0),
        # Kullanici, 24 Eylul: "C1 C2 C3 farklı noktalara düşsün ki farklı vektörler etki etsin"
        # ve "şu an 0.9 ile başlayabiliriz".
        'TS_PV_LAM09': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0,
                            lam=0.9),
        # Egitimsiz kNN: absolute lam 0,9 %13,3, relative lam 0,7 %34,8.  Kullanici: "Uygun".
        'TS_PV_REL07': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0,
                            lam=0.7, chain='relative'),
        # Kagit ustu (REL07 t2000, sabit g 0,1): ppl 30,3 -> 25,9.  Kullanici: "c_cche ve gate olsun".
        'TS_PV_CCACHE': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0,
                             lam=0.7, chain='relative', c_cache=True, cache_topk=8, cache_skip=3,
                             s_c_init=3.0, gate_0_init=-2.0),
        # Kagit ustu (CCACHE t8000, yalniz aramada skip): 3 -> 20 dongu olcusu 0,28 -> 0,63, ppl 19,0 -> 19,7.
        # Kullanici, 24 Eylul: "durdur ve skip 20 ile eğitelim".
        'TS_PV_CCSKIP20': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0,
                               lam=0.7, chain='relative', c_cache=True, cache_topk=8, cache_skip=20,
                               s_c_init=3.0, gate_0_init=-2.0),
        # Kagit ustu (CCACHE t10000, 150 tutulan): arama mukemmel olsa accuracy %40,3 -> %66,7; filtre
        # donguyu kirmadi, defter kapaliyken de 19/20 dongu (kok: C'de yalniz son ~5 kelime).
        # Kullanici, 24 Eylul: "query (Q_t)", "64 ok, 8 aktif", "512/512 mantıklı",
        # "lam_w mantıklı beta_w mantıklı tam boyut olsun".
        # Kagit ustu (CCACHE t10000, egitimsiz): tasinmis noktanin boyu secimi eziyor, derin katmanlarda
        # 10 / 6 vektor, C_m 40 yon; secim yone bakinca ayni vektorlerle C_m 113 yon.  CCACHE'in AYNISI,
        # tek fark select.  Kullanici, 24 Eylul: "b'yi ekle, adı SELECT olsun, ayrı koşu yapalım".
        'TS_PV_SELECT': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512,
                             start_norm=1.0, lam=0.7, chain='relative', c_cache=True, cache_topk=8,
                             cache_skip=3, s_c_init=3.0, gate_0_init=-2.0, select='direction'),
        # CCACHE'in AYNISI, tek fark active 256: nokta butun vektorlerden gecer (top-8 yok).  4.000 adim.
        # Kullanici, 24 Eylul: "Önce kısa bir tek-değişken koşusu. CCACHE'in aynısı, tek fark active=256".
        'TS_PV_DENSE': dict(SADE, d_order=1024, d_content=0, vectors=256, active=256, layers=4, t_max=512,
                            start_norm=1.0, lam=0.7, chain='relative', c_cache=True, cache_topk=8,
                            cache_skip=3, s_c_init=3.0, gate_0_init=-2.0),
        # Secim sagligi (t2000): hepsi aktif + uzaklik start'lari merkeze yapistirdi, top-8 derinde 4-26 vektor.
        # SELECT'in AYNISI, tek fark load_balance 0,01.  4.000 adim.
        # Kullanici, 24 Eylul: "load balancing ekle, select ile birlikte koş".
        'TS_PV_SELECT_LB': dict(SADE, d_order=1024, d_content=0, vectors=256, active=8, layers=4, t_max=512,
                                start_norm=1.0, lam=0.7, chain='relative', c_cache=True, cache_topk=8,
                                cache_skip=3, s_c_init=3.0, gate_0_init=-2.0, select='direction',
                                load_balance=0.01),
        # Kagit ustu (SELECT_LB t4000, yakin C ciftleri): farkli kelime isteyen yakin ciftlerin %80'inde son 4
        # kelime ayni -- bilgi C'de yok (bellek).  SELECT_LB'nin AYNISI, tek fark c_content: C = [C_order 512 |
        # C_content 512], lam_w / beta_w ogrenilir.  4.000 adim.  Kullanici, 24 Eylul: "Select b content geç".
        'TS_PV_LB_CONTENT': dict(SADE, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0, lam=0.7,
                                 chain='relative', c_cache=True, cache_topk=8, cache_skip=3, s_c_init=3.0,
                                 gate_0_init=-2.0, select='direction', load_balance=0.01, c_content=True,
                                 d_order=512, d_content=512),
        # LB_CONTENT 4.000 (f1ff3ed): heldout 0,4434 / ppl 14,35, son katman 32 vektor.  AYNI ayarlar, YENI kod
        # (a82f5d6: denge P'si sabit sicaklikla, gunlukte NLL), 0'dan 20.000 adim.
        # Kullanici, 24 Eylul: "yeni hali ile koşalım 0 dan 20.000 adım".
        'TS_PV_LB_CONTENT2': dict(SADE, vectors=256, active=8, layers=4, t_max=512, start_norm=1.0, lam=0.7,
                                  chain='relative', c_cache=True, cache_topk=8, cache_skip=3, s_c_init=3.0,
                                  gate_0_init=-2.0, select='direction', load_balance=0.01, c_content=True,
                                  d_order=512, d_content=512),
        # TEPE: model_18'in son yapisi, hicbir deger degismez: butun vektorler aktif, secim yone gore,
        # D_SUM 1024 = 512 + 512, defter butun gecmis + Q 64 ok (hepsi aktif), C_content lam_w/beta_w.
        # Kullanici, 24 Eylul: "evet katılıyorum, select'i durdur ve tepeyi kur".
        'TS_PV_FULL': dict()}
EK = dict(veri='tinystories tam_n4000_v2', iz=DS.iz(SOZ, EG[0][:1000].numpy()),
          T=512, kod=KOD)


def bellek_gb(d_order=model_18.D_ORDER, d_content=model_18.D_CONTENT, vectors=model_18.VECTORS,
              active=model_18.ACTIVE, layers=model_18.LAYERS, t_max=None, c_cache=model_18.C_CACHE,
              cache_topk=model_18.CACHE_TOPK, query=model_18.QUERY, query_vectors=model_18.QUERY_VECTORS,
              query_active=model_18.QUERY_ACTIVE, c_content=model_18.C_CONTENT, **ayar):
    """Ileri gecisin geri yayilim icin SAKLADIGI, TAHMIN: layer basina uzaklik tablosu,
    aktif vektorler, C; N noktalik skor tablosu (x3); c_cache: p_cache ve karisim (x4) + (T,T);
    butun gecmis (cache_topk None): siralama, W_c, ids (x6 (T,T)); Q oklari; C_content parcalari (x6).
    Olculdu (REL07, 24 Eylul): tahmin 7,6 GB, nvidia-smi 5,2 GB -- tahmin fazla."""
    B, T = ORTAK['batch'], EG[0].shape[1]
    d = d_order + d_content                                   # D_SUM
    ek = 4 * B * T * N + B * T * T if c_cache else 0
    ek += 6 * B * T * T if c_cache and cache_topk is None else 0
    katman = lambda v, a: 2 * v + 3 * d if a >= v else v + a * d + 3 * d   # hepsi aktif: (B,T,v) + matris carpimi
    ek += B * T * katman(query_vectors, query_active) if query else 0
    ek += 6 * B * T * d_content if c_content else 0
    return (layers * B * T * katman(vectors, active) + 3 * B * T * N + ek) * 4 / 1e9


def son_model(ad):
    """Diskteki EN YENI agirlik (w ya da t) -- kosu surerken de, ayri bir kopya."""
    d = KOK + '/' + ad
    f = max((int(re.findall('[0-9]+', x)[0]), x) for x in os.listdir(d)
            if re.match('[tw][0-9]+[.]pt$', x))[1]
    k = torch.load(d + '/' + f, weights_only=False, map_location='cuda')
    m = model_18.PV.from_package(k).cuda()
    return m, k


print('sozluk %s token   <eos> %d   egitim %s pencere   tutulan %s pencere   T=%d'
      % (f'{N:,}', EOS, f'{len(EG[0]):,}', f'{len(DG[0]):,}', EG[0].shape[1]))
print('1 epok = %s adim   %s adim = %.2f epok'
      % (f"{len(EG[0]) // ORTAK['batch']:,}", f"{ORTAK['steps']:,}",
         ORTAK['steps'] * ORTAK['batch'] / len(EG[0])))
for ad, a in KOSU.items():
    _m = model_18.PV(N, **a)
    par = sum(p.numel() for p in _m.parameters())
    print('%-14s D_SUM %d  vectors %d  active %d  layers %d  skor %s   parametre %s   saklanan ~%.2f GB'
          % (ad, _m.d_sum, _m.vectors, _m.active, _m.layers,
             '-%s*%s' % ('e^s' if _m.S_p_learned else '%g' % _m.S_p,
                         'D^2' if _m.squared else 'D'), f'{par:,}', bellek_gb(**a)))
print('\nORNEK PENCERE\n' + DS.coz(EG[0][0][EG[1][0]].numpy(), SOZ)[:600])

In [ ]:
# 1 KOSU BASLAT -- PV hikaye  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1 = 'TS_PV_D1024'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_1]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1], **ORTAK))

In [ ]:
# 1b KOSU BASLAT -- PV hikaye, start'lar KUCUK (start_norm 1)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1B = 'TS_PV_START1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_1B]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1B, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1B], **ORTAK))

In [ ]:
# 1c KOSU BASLAT -- PV hikaye, zincir solar (lam 0,9) + start_norm 1  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1C = 'TS_PV_LAM09'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_1C]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1C, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1C], **ORTAK))

In [ ]:
# 1d KOSU BASLAT -- PV hikaye, GORELI zincir (lam 0,7) + start_norm 1  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1D = 'TS_PV_REL07'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_1D]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1D, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1D], **ORTAK))

In [ ]:
# 1e KOSU BASLAT -- PV hikaye, GORELI zincir + C_cache ve gate  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1E = 'TS_PV_CCACHE'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD_1E]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1E, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1E], **ORTAK))

In [ ]:
# 1f KOSU BASLAT -- PV hikaye, C_cache son 20 kelimeyi aramaz (cache_skip 20)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1F = 'TS_PV_CCSKIP20'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD_1F]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1F, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1F], **ORTAK))

In [ ]:
# 1g KOSU BASLAT -- PV hikaye, TEPE: hepsi aktif + yon secimi + Q + C_content  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1G = 'TS_PV_FULL'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD_1G]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1G, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1G], **ORTAK))

In [ ]:
# 1h KOSU BASLAT -- PV hikaye, CCACHE + vektor secimi YONE gore (select direction)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1H = 'TS_PV_SELECT'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD_1H]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Hedef: dolgu disinda her kelime.
print(train.start(AD_1H, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1H], **ORTAK))

In [ ]:
# 1i KOSU BASLAT -- PV hikaye, CCACHE + butun vektorler aktif (active 256), 4.000 adim  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1I = 'TS_PV_DENSE'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD_1I]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Kisa tek-degisken kosusu: 4.000 adim.
print(train.start(AD_1I, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1I], **dict(ORTAK, steps=4000)))

In [ ]:
# 1j KOSU BASLAT -- PV hikaye, SELECT + load balancing (0,01), 4.000 adim  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1J = 'TS_PV_SELECT_LB'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).  Denge: katman basina (B,T,256) P.
_ger = max(2.0, 1.5 * (bellek_gb(**KOSU[AD_1J]) + 4 * ORTAK['batch'] * EG[0].shape[1] * 256 * 4 / 1e9))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Kisa tek-degisken kosusu: 4.000 adim.
print(train.start(AD_1J, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1J], **dict(ORTAK, steps=4000)))

In [ ]:
# 1k KOSU BASLAT -- PV hikaye, SELECT_LB + C_content (lam_w, beta_w), 4.000 adim  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1K = 'TS_PV_LB_CONTENT'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).  Denge: katman basina (B,T,256) P.
_ger = max(2.0, 1.5 * (bellek_gb(**KOSU[AD_1K]) + 4 * ORTAK['batch'] * EG[0].shape[1] * 256 * 4 / 1e9))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Kisa tek-degisken kosusu: 4.000 adim.
print(train.start(AD_1K, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1K], **dict(ORTAK, steps=4000)))

In [ ]:
# 1l KOSU BASLAT -- PV hikaye, LB_CONTENT yeni kodla (sabit sicaklikli denge), 20.000 adim  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
AD_1L = 'TS_PV_LB_CONTENT2'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).  Denge: katman basina (B,T,256) P.
_ger = max(2.0, 1.5 * (bellek_gb(**KOSU[AD_1L]) + 4 * ORTAK['batch'] * EG[0].shape[1] * 256 * 4 / 1e9))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  20.000 adim (kural 1: ilk sinir).
print(train.start(AD_1L, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, **KOSU[AD_1L], **dict(ORTAK, steps=20000)))

In [ ]:
# 2 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin ADIM'i buyut, bu hucreyi calistir.
AD = 'TS_PV_LB_CONTENT2'
ADIM = 20000                       # kosunun hedefi (1l); ORTAK['steps'] 16.000

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
import gc
gc.collect()
torch.cuda.empty_cache()           # durdurulan kosunun onbellegi bos sayilsin
_bos = torch.cuda.mem_get_info()[0] / 1e9
# Pay 1,5: olculdu, REL07 tahmin 7,6 GB, nvidia-smi 5,2 GB (24 Eylul).
_ger = max(2.0, 1.5 * bellek_gb(**KOSU[AD]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

_d = KOK + '/' + AD
_n = sorted((int(re.findall('[0-9]+', f)[0]), f)
            for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
SON = _d + '/' + _n[-1][1]
print('son nokta  %s   adim %s   hedef %s' % (SON, f'{_n[-1][0]:,}', f'{ADIM:,}'))
assert _n[-1][0] < ADIM, 'zaten hedefe varmis -- uzatmak icin ADIM buyut'

print(train.start(AD, (EG[0], EG[1], EG[1]), N, metric=OLCUT, device='cuda', root=KOK,
                  extra=EK, vocab=SOZ, resume=SON, **KOSU[AD], **dict(ORTAK, steps=ADIM)))

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu basar.
# Bellekteki BUTUN kosu nesnelerine bakar (HAZIRLIK modulleri yeniden yuklese de eski kosu gorunur):
# CANLI olanlar; hic canli yoksa en son baslayani.
import gc
_runs = [o for o in gc.get_objects() if type(o).__name__ == 'Run']
_goster = [r for r in _runs if r.alive] or sorted(_runs, key=lambda r: len(r.log))[-1:]
for _r in _goster:
    print('%s: %s   gunluk %d satir%s' % (_r.name, 'CANLI' if _r.alive else 'bitti', len(_r.log),
                                          '   DURDUR istendi' if _r.stop_requested else ''))
    for _s in _r.log[-30:]:
        print(_s)

In [ ]:
# 4 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Ust: accuracy (her 500 adim, 2.000 + 2.000 pencere).  Alt: HER ADIMIN kaybi.
import matplotlib.pyplot as plt


def satirlar(ad):
    """gunluk.txt -> {step: (loss, train, heldout)}."""
    r, yol = {}, KOK + '/' + ad + '/gunluk.txt'
    if os.path.exists(yol):
        for s in open(yol, encoding='utf-8'):
            p = s.split()
            if len(p) >= 6 and p[1].isdigit():
                try:
                    r[int(p[1])] = tuple(float(x) for x in p[2:5])
                except ValueError:
                    pass
    return r


def step_losses(ad):
    """Canli kosudan (train.RUNS) ya da diskteki son paketten."""
    k = train.RUNS[ad].result.get('step_losses') if ad in train.RUNS else None
    yol = KOK + '/model_' + ad + '.pt'
    if k is None and os.path.exists(yol):
        try:
            k = torch.load(yol, weights_only=False, map_location='cpu').get('step_losses')
        except Exception as h:          # yazilirken okunduysa
            print('  %s paketi okunamadi (%s) -- tekrar dene' % (ad, h))
    return None if k is None else k[~k.isnan()]


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ad in KOSU:
    r, k = satirlar(ad), step_losses(ad)
    if r:
        x = sorted(r)
        a1.plot(x, [r[i][2] for i in x], label=ad + ' heldout')
        a1.plot(x, [r[i][1] for i in x], ':', label=ad + ' train')
        print('%-14s %d nokta   son adim %s  heldout accuracy %.4f'
              % (ad, len(x), f'{x[-1]:,}', r[x[-1]][2]))
    if k is not None and len(k) > 200:
        a2.plot(k.numpy(), lw=0.4, label=ad)
a1.set_ylabel('accuracy (sonraki kelime)')
a1.legend(fontsize=7)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 HIKAYE YAZ  |  GPU  |  tekrar: GUVENLI -- diskteki EN YENI agirlikla, kosu surerken de
# SAYI degil METIN: model istemin devamini yazar, <eos> uretince durur.
AD = 'TS_PV_LB_CONTENT2'
ISTEM_SAYISI, SICAKLIK = 4, (0.0, 0.8)

# --- GPU KAPISI (CLAUDE.md kural 2)
import random, textwrap
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
print('GPU kapisi GECTI: ' + torch.cuda.get_device_name(0))

m, k = son_model(AD)
print('%s   adim %s   heldout accuracy %.4f' % (AD, f"{k['step']:,}", k['heldout_acc']))
for istem in random.Random(0).sample(DS.istemler(TS), ISTEM_SAYISI):
    for s in SICAKLIK:
        bas, yazdi = DS.devam(m, istem, SOZ, IX, adim=150, aygit='cuda', sicaklik=s, tohum=0)
        print('\n' + '-' * 72 + '\nISTEM   ' + '\n        '.join(textwrap.wrap(bas, 64)))
        print('MODEL (sicaklik %.1f)\n        ' % s + '\n        '.join(textwrap.wrap(yazdi, 64)))

In [ ]:
# 6 SONUC  |  GPU  |  tekrar: GUVENLI -- kosu BITTIKTEN sonra, tutulan pencerelerin TAMAMI
import math
AD = 'TS_PV_LB_CONTENT2'

# --- GPU KAPISI (CLAUDE.md kural 2)
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
print('GPU kapisi GECTI: ' + torch.cuda.get_device_name(0))
assert not (AD in train.RUNS and train.RUNS[AD].alive), AD + ' HALA KOSUYOR'

m, k = son_model(AD)
print('%s   adim %s   (TAM) train %.4f   heldout %.4f   ppl %.2f'
      % (AD, f"{k['step']:,}", k['train_acc'], k['heldout_acc'], math.exp(k['heldout_ce']))
      if k.get('done') else '%s   adim %s  -- BITMEMIS' % (AD, f"{k['step']:,}"))
print('\nKONUMA GORE accuracy (tutulan): C hikayenin neresinde doyuyor')
DS.konum_tablo(m, DG[0], DG[1], aygit='cuda')

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
train.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for ad in KOSU:
    _d = KOK + '/' + ad
    if not os.path.isdir(_d):
        print('%-14s henuz kayit yok' % ad)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f)
    print('%-14s %3d yedek   %.3f GB   %s'
          % (ad, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi

In [ ]:
# W IPLIKLER  |  CPU  |  tekrar: GUVENLI -- HAZIRLIK modulleri yeniden yukleyince eski kosu listesi
# gorunmez olur; bellekteki BUTUN kosu nesnelerini listeler (yeni + eski modul).
# DURDUR_KOPYA: ayni adla IKI canli kosu varsa, gunlugu KISA olani (sonradan baslayan kopya) durdurur.
import gc
DURDUR_KOPYA = False
_runs = [o for o in gc.get_objects() if type(o).__name__ == 'Run']
for _r in _runs:
    print('%-20s canli %-5s  durdur istendi %-5s  gunluk %3d satir  son: %s'
          % (_r.name, _r.alive, _r.stop_requested, len(_r.log), (_r.log[-1] if _r.log else '')[:90]))
if DURDUR_KOPYA:
    for _ad in {r.name for r in _runs}:
        _canli = sorted([r for r in _runs if r.name == _ad and r.alive], key=lambda r: len(r.log))
        for _r in _canli[:-1]:
            _r.stop()
            print('DURDURULDU (kopya): %s  gunluk %d satir' % (_r.name, len(_r.log)))